# Notebook 7: Econometric Results and Diagnostics

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Research Period:** April 2017 – December 2025

## Notebook Objective

This notebook evaluates the statistical adequacy and reliability of the econometric models developed in Notebook 6.

The analysis focuses on:

1. reproducing the selected ARDL and NARDL specifications
2. assessing residual serial correlation
3. testing for heteroskedasticity
4. examining residual normality
5. evaluating parameter and model stability
6. reviewing models with diagnostic failures
7. consolidating the long-run and short-run findings
8. exporting validated econometric results for later comparison with the machine learning models.

No new lag selection is performed in this notebook. The specifications selected
in Notebook 6 are treated as the starting point for diagnostic assessment.

## Diagnostic Context

Statistically significant coefficients do not necessarily imply that a model is adequately specified.

ARDL and NARDL inference depends on assumptions concerning the behaviour of the model residuals and the stability of the estimated relationship. Diagnostic testing is therefore required before the estimated pass-through effects are treated as final research findings.

The diagnostic process distinguishes between:

- a statistically estimated relationship;
- a relationship that passes the relevant diagnostic tests; and
- a relationship that remains useful but requires a methodological caveat.

This distinction prevents coefficient significance from being interpreted without considering the reliability of the underlying model.

## Diagnostic Framework

The selected econometric models are assessed using the following diagnostic areas:

### Serial correlation

Residual serial correlation indicates that the model has not fully captured the time-dependent structure of the series.

### Heteroskedasticity

Heteroskedastic residuals have non-constant variance and may affect the
reliability of conventional standard errors and hypothesis tests.

### Residual normality

Normality is assessed because strongly non-normal residuals may influence small-sample statistical inference. Normality is treated as a supporting diagnostic rather than an automatic model-rejection rule.

### Functional form

Functional-form assessment examines whether important nonlinear structure may remain unexplained by the model.

### Parameter stability

Stability tests assess whether the estimated relationship remains reasonably consistent across the modelling period.

A 5% significance level is used unless otherwise stated. Diagnostic failures are reported transparently and considered jointly rather than using one test as an automatic reason to discard a model.

In [ ]:
# import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statsmodels.tsa.ardl import ARDL, UECM

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("Libraries imported successfully.")

## Load Econometric Data and Results

The category-level econometric dataset and the result tables exported by Notebook 6 are loaded from their established project locations.

The exported lag-selection tables provide the specifications required to re-estimate the models. The remaining tables preserve the bounds-test, error-correction, coefficient and asymmetry findings that will be evaluated against the diagnostic results.

In [ ]:
# define input locations
econometric_data_path = Path(
    "../data/processed/econometric_model_data.csv"
)

econometric_results_directory = Path(
    "../reports/tables/econometrics"
)

result_file_names = [
    "symmetric_lag_selection.csv",
    "asymmetric_lag_selection.csv",
    "long_run_bic_comparison.csv",
    "final_bounds_results.csv",
    "error_correction_results.csv",
    "long_run_effects.csv",
    "long_run_asymmetry_results.csv",
    "symmetric_short_run_selection.csv",
    "asymmetric_short_run_selection.csv",
    "short_run_bic_comparison.csv",
    "short_run_asymmetry_results.csv",
]

required_paths = [
    econometric_data_path,
    *[
        econometric_results_directory / file_name
        for file_name in result_file_names
    ],
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_path_list = "\n".join(
        str(path) for path in missing_paths
    )
    raise FileNotFoundError(
        f"Required Notebook 6 outputs are missing:\n{missing_path_list}"
    )

print("All required input files are available.")

In [ ]:
# load the econometric dataset
econometric_data = pd.read_csv(
    econometric_data_path,
    parse_dates=["Date"],
)

# Load the exported result tables
econometric_result_tables = {
    Path(file_name).stem: pd.read_csv(
        econometric_results_directory / file_name
    )
    for file_name in result_file_names
}

symmetric_lag_selection = econometric_result_tables[
    "symmetric_lag_selection"
]
asymmetric_lag_selection = econometric_result_tables[
    "asymmetric_lag_selection"
]
final_bounds_results = econometric_result_tables[
    "final_bounds_results"
]
error_correction_results = econometric_result_tables[
    "error_correction_results"
]
long_run_effects = econometric_result_tables[
    "long_run_effects"
]
long_run_asymmetry_results = econometric_result_tables[
    "long_run_asymmetry_results"
]
symmetric_short_run_selection = econometric_result_tables[
    "symmetric_short_run_selection"
]
asymmetric_short_run_selection = econometric_result_tables[
    "asymmetric_short_run_selection"
]
short_run_asymmetry_results = econometric_result_tables[
    "short_run_asymmetry_results"
]

print("Econometric data and result tables loaded successfully.")

In [ ]:
# validate the Notebook 6 handoff
input_validation = pd.Series(
    {
        "Econometric observations": len(econometric_data),
        "Food subclasses": econometric_data[
            "SubclassDescription"
        ].nunique(),
        "Unique months": econometric_data["Date"].nunique(),
        "Duplicate subclass-month rows": econometric_data.duplicated(
            subset=["SubclassDescription", "Date"]
        ).sum(),
        "Missing econometric values": int(
            econometric_data.isna().sum().sum()
        ),
        "Result tables loaded": len(econometric_result_tables),
        "Symmetric specifications": len(
            symmetric_lag_selection
        ),
        "Asymmetric specifications": len(
            asymmetric_lag_selection
        ),
        "Final bounds-test results": len(
            final_bounds_results
        ),
    },
    name="Value",
).to_frame()

display(input_validation)

result_table_manifest = pd.DataFrame(
    [
        {
            "Table": table_name,
            "Rows": result_table.shape[0],
            "Columns": result_table.shape[1],
        }
        for table_name, result_table
        in econometric_result_tables.items()
    ]
)

display(result_table_manifest)

## Re-estimate Selected Models

The selected models are re-estimated using the lag orders exported by Notebook 6.

Four model collections are reconstructed:

1. symmetric ARDL level models
2. asymmetric NARDL level models
3. symmetric stationary short-run models
4. asymmetric stationary short-run models

The same dependent variables, exchange-rate variables, seasonal indicators and six-month hold-back period used during model selection are retained. This ensures that the diagnostic tests are applied to the exact specifications from which the reported results were obtained.

Re-estimation also makes the model residuals and fitted values available within the current notebook without relying on temporary Python objects from Notebook 6.